# Regularized Regression — Ridge, Lasso & ElasticNet

The [linear regression chapter](linear-regression.ipynb) fit an unpenalized model
(ordinary least squares). With few, clean features that's fine — but on real data
with **many features, some irrelevant or correlated**, OLS happily assigns weight
to noise and overfits. **Regularization** adds a penalty on coefficient size to
counter that:

- **Ridge** (L2) shrinks all coefficients toward zero, but never exactly to zero.
- **Lasso** (L1) can drive coefficients *exactly to zero* — it does feature
  selection for you.
- **ElasticNet** blends the two.

```{note}
The linear-regression chapter used `linfa`, but `linfa-linear` has no Ridge/Lasso.
We switch to [`smartcore`](https://docs.rs/smartcore), which bundles all three
regularized regressors in one crate with a uniform API (and is already pre-warmed).
```

To make the effect unmistakable we use a **synthetic** dataset where we *know* the
truth: `y = 3·x0 − 2·x1 + noise`, with four extra features (`x2…x5`) that are pure
noise. A good regularizer should ignore `x2…x5`.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::{Array, Array2};
use smartcore::model_selection::train_test_split;
use smartcore::metrics::mean_squared_error;
use plotters::prelude::*;

// Synthetic data: only x0, x1 matter. Fixed-seed LCG (inlined, no closure) so the
// dataset is reproducible and the persisted values are plain Vecs.
let (data, yv): (Vec<f32>, Vec<f32>) = {
    let (n, p) = (150usize, 6usize);
    let mut seed = 7u64;
    let mut data = Vec::with_capacity(n * p);
    let mut yv = Vec::with_capacity(n);
    for _ in 0..n {
        let mut row = [0f64; 6];
        for j in 0..p {
            seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
            row[j] = ((seed >> 11) as f64) / ((1u64 << 53) as f64) * 2.0 - 1.0;
        }
        seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let noise = (((seed >> 11) as f64) / ((1u64 << 53) as f64) * 2.0 - 1.0) * 0.3;
        yv.push((3.0 * row[0] - 2.0 * row[1] + noise) as f32);
        for j in 0..p { data.push(row[j] as f32); }
    }
    (data, yv)
};
let p = 6usize;
let x: DenseMatrix<f32> = DenseMatrix::new(yv.len(), p, data.clone(), false);
println!("{} samples x {} features (true coefficients: x0=+3, x1=-2, x2..x5=0)", yv.len(), p);

## Fit all four, on one split

We fit OLS, Ridge, Lasso, and ElasticNet on the *same* train/test split (fixed
seed), then collect each model's coefficients and its held-out RMSE. The `alpha`
values set the penalty strength (Lasso/ElasticNet also has an `l1_ratio` mixing
L1 vs L2).

In [ ]:
use smartcore::linear::linear_regression::LinearRegression;
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};
use smartcore::linear::lasso::{Lasso, LassoParameters};
use smartcore::linear::elastic_net::{ElasticNet, ElasticNetParameters};

// Coefficients come back as a DenseMatrix; flatten to Vec<f32> regardless of shape.
fn coefs(m: &DenseMatrix<f32>) -> Vec<f32> {
    let (r, c) = m.shape();
    let nn = r.max(c);
    (0..nn).map(|k| if r >= c { *m.get((k, 0)) } else { *m.get((0, k)) }).collect()
}

let (ols_c, ridge_c, lasso_c, enet_c, rmses):
    (Vec<f32>, Vec<f32>, Vec<f32>, Vec<f32>, Vec<(String, f64)>) = {
    let (xtr, xte, ytr, yte) = train_test_split(&x, &yv, 0.3, true, Some(1));
    let ols   = LinearRegression::fit(&xtr, &ytr, Default::default()).unwrap();
    let ridge = RidgeRegression::fit(&xtr, &ytr,
                    RidgeRegressionParameters::default().with_alpha(1.0_f32)).unwrap();
    let lasso = Lasso::fit(&xtr, &ytr,
                    LassoParameters::default().with_alpha(0.2)).unwrap();
    let enet  = ElasticNet::fit(&xtr, &ytr,
                    ElasticNetParameters::default().with_alpha(0.2).with_l1_ratio(0.5)).unwrap();
    let rmse = |name: &str, pred: Vec<f32>| (name.to_string(), mean_squared_error(&yte, &pred).sqrt());
    let rmses = vec![
        rmse("OLS",        ols.predict(&xte).unwrap()),
        rmse("Ridge",      ridge.predict(&xte).unwrap()),
        rmse("Lasso",      lasso.predict(&xte).unwrap()),
        rmse("ElasticNet", enet.predict(&xte).unwrap()),
    ];
    (coefs(ols.coefficients()), coefs(ridge.coefficients()),
     coefs(lasso.coefficients()), coefs(enet.coefficients()), rmses)
};

println!("test RMSE:");
for (name, r) in &rmses { println!("  {:>11}: {:.3}", name, r); }
println!("\ncoefficients (x0..x5):");
println!("  {:>11}: {:?}", "OLS",        ols_c.iter().map(|v| (v*100.0).round()/100.0).collect::<Vec<_>>());
println!("  {:>11}: {:?}", "Ridge",      ridge_c.iter().map(|v| (v*100.0).round()/100.0).collect::<Vec<_>>());
println!("  {:>11}: {:?}", "Lasso",      lasso_c.iter().map(|v| (v*100.0).round()/100.0).collect::<Vec<_>>());
println!("  {:>11}: {:?}", "ElasticNet", enet_c.iter().map(|v| (v*100.0).round()/100.0).collect::<Vec<_>>());

## The sparsity effect, visualised

A grouped bar chart of each model's coefficient per feature. Watch `x2…x5` (the
noise features): **OLS and Ridge** leave them small-but-nonzero, while **Lasso**
(and ElasticNet, less aggressively) push them to **exactly zero** — recovering the
true sparse structure. The two real features (`x0≈+3`, `x1≈−2`) survive in all.

In [ ]:
// Wrapped in a block: `models` holds references, which evcxr can't persist
// across cells — keeping it local sidesteps that (the figure is still returned).
{
let models: [(&str, &Vec<f32>, RGBColor); 4] = [
    ("OLS",        &ols_c,   RGBColor(150, 150, 150)),
    ("Ridge",      &ridge_c, RGBColor(30, 90, 200)),
    ("Lasso",      &lasso_c, RGBColor(210, 50, 50)),
    ("ElasticNet", &enet_c,  RGBColor(40, 160, 80)),
];
let all: Vec<f64> = models.iter().flat_map(|(_, c, _)| c.iter().map(|&v| v as f64)).collect();
let ymin = all.iter().cloned().fold(0.0, f64::min) - 0.4;
let ymax = all.iter().cloned().fold(0.0, f64::max) + 0.4;

evcxr_figure((720, 420), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("coefficient by feature and model", ("sans-serif", 16))
        .margin(10).x_label_area_size(34).y_label_area_size(44)
        .build_cartesian_2d(0f64..(p as f64), ymin..ymax)?;
    chart.configure_mesh().x_desc("feature index").y_desc("coefficient")
        .x_labels(p).x_label_formatter(&|v| format!("x{}", *v as usize)).draw()?;
    chart.draw_series(std::iter::once(PathElement::new(vec![(0.0, 0.0), (p as f64, 0.0)], BLACK)))?;
    for (mi, (name, cv, color)) in models.iter().enumerate() {
        let color = *color;
        chart.draw_series((0..p).map(|j| {
            let x0 = j as f64 + 0.05 + mi as f64 * 0.225;
            Rectangle::new([(x0, 0.0), (x0 + 0.2, cv[j] as f64)], color.filled())
        }))?
        .label(*name)
        .legend(move |(x, y)| Rectangle::new([(x, y - 4), (x + 10, y + 4)], color.filled()));
    }
    chart.configure_series_labels().position(SeriesLabelPosition::UpperRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})
}

## The regularization path

The bar chart above is one snapshot at a single `alpha`. The **regularization
path** sweeps `alpha` across orders of magnitude and traces each coefficient —
the canonical way to watch Lasso zero out the noise features one by one while the
two real ones (`x0≈+3`, `x1≈−2`) hold on longest. `plotters-statistical`'s
`RegularizationPath` draws it, one coloured line per feature on a log-scale
strength axis:

In [ ]:
use plotters_statistical::RegularizationPath;
{
    // Sweep Lasso alpha on a log grid; collect coefficients[alpha][feature].
    let alphas: Vec<f64> = (0..30).map(|i| 0.005 * 1.25_f64.powi(i)).collect();
    let path_coefs: Vec<Vec<f64>> = alphas.iter().map(|&a| {
        let m = Lasso::fit(&x, &yv, LassoParameters::default().with_alpha(a)).unwrap();
        coefs(m.coefficients()).iter().map(|&v| v as f64).collect()
    }).collect();

    evcxr_figure((720, 420), |root| {
        root.fill(&WHITE)?;
        let mut chart = ChartBuilder::on(&root)
            .caption("Lasso regularization path", ("sans-serif", 16))
            .margin(10).x_label_area_size(34).y_label_area_size(44)
            .build_cartesian_2d((alphas[0]..alphas[alphas.len() - 1]).log_scale(), -2.5f64..3.5f64)?;
        chart.configure_mesh().x_desc("alpha (log)").y_desc("coefficient").draw()?;
        let path = RegularizationPath::new(&alphas, &path_coefs)?
            .feature_names(["x0", "x1", "x2", "x3", "x4", "x5"].iter().copied())
            .stroke_width(2);
        for line in path.lines() {
            let color = line.color();
            let name = line.name().unwrap_or_default().to_string();
            chart.draw_series(std::iter::once(line))?
                .label(name)
                .legend(move |(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], color));
        }
        chart.configure_series_labels().position(SeriesLabelPosition::UpperRight)
            .border_style(BLACK).background_style(WHITE.mix(0.85)).draw()?;
        Ok(())
    })
}

## Choosing between them

| Model | Penalty | Effect | Use when |
| --- | --- | --- | --- |
| **OLS** | none | fits everything, incl. noise | few, clean, relevant features |
| **Ridge** | L2 | shrinks all coefficients | many correlated features, keep them all |
| **Lasso** | L1 | zeroes weak coefficients (sparse) | you want automatic feature selection |
| **ElasticNet** | L1 + L2 | sparse *and* handles correlation | many features, some correlated |

`alpha` (penalty strength) is itself a hyperparameter — too small barely
regularizes, too large underfits. Choose it the honest way: a search scored by
**cross-validation** (the [Model Evaluation](../01d-evaluation/cross-validation.ipynb)
chapter), which the [Optimization](../05b-optimization/hyperparameter-search.ipynb)
chapter automates. The [learning/validation curves](../01d-evaluation/learning-curves.ipynb)
chapter's "high variance → regularization helps" diagnosis is exactly the problem
this chapter's tools solve.

Next: [classification models](../02b-classification/knn-classification.ipynb) — a
different task type (predicting categories, not quantities).